In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [3]:
df = pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/course/datasets/data_coindesk_base.csv')
df = df[['SENTIMENT', 'PUBLISHED_ON']][::-1] # reverse

def sent(x):
    if x == 'POSITIVE':
        return 1
    elif x == 'NEGATIVE':
        return -1
    else:
        return 0

df['SENTIMENT'] = df['SENTIMENT'].apply(sent)
df['PUBLISHED_ON'] = pd.to_datetime(df['PUBLISHED_ON'], unit='s')
df.dropna(inplace=True)

In [4]:
min_date = min(df['PUBLISHED_ON'])
max_date = max(df['PUBLISHED_ON'])
min_date, max_date

(Timestamp('2023-10-26 17:41:25'), Timestamp('2025-11-14 11:43:09'))

In [5]:
df.head()

,SENTIMENT,PUBLISHED_ON
182499,1,2023-10-26 17:41:25
182498,0,2023-10-26 17:58:21
182497,0,2023-10-26 18:00:00
182496,1,2023-10-26 18:00:00
182495,1,2023-10-26 18:00:12


теперь берем данные с биржи

In [6]:
!pip install -q ccxt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 641.1/641.1 kB 31.1 MB/s eta 0:00:00


In [7]:
import ccxt
import time
from tqdm.auto import tqdm
exchange = ccxt.kucoin({
    'enableRateLimit': True
})
print(exchange.timeframes)

{'1m': '1min', '3m': '3min', '5m': '5min', '15m': '15min', '30m': '30min', '1h': '1hour', '2h': '2hour', '4h': '4hour', '6h': '6hour', '8h': '8hour', '12h': '12hour', '1d': '1day', '1w': '1week', '1M': '1month'}


In [8]:
prices = []

symbol = 'BTC/USDT'
tf = '1h'
since = exchange.parse8601('2023-10-26T17:41:25Z')

while True:
    batch = exchange.fetch_ohlcv(symbol, tf, since=since, limit=1000)
    if not batch:
        break
    prices.extend(batch)
    since = batch[-1][0] + 60_000 * 60   # + 1 час
    time.sleep(exchange.rateLimit / 1000)

df_prices = pd.DataFrame(prices, columns=[
    'timestamp', 'open', 'high', 'low', 'close', 'volume'
])
df_prices['timestamp'] = pd.to_datetime(df_prices['timestamp'], unit='ms')

[1698343200000, 33944.5, 34084.0, 33938.0, 33983.2, 87.65195539] - это batch

In [9]:
df_prices.head()

,timestamp,open,high,low,close,volume
0,2023-10-26 18:00:00,33944.5,34084.0,33938.0,33983.2,87.651955
1,2023-10-26 19:00:00,33983.2,34108.3,33909.1,34040.3,128.877931
2,2023-10-26 20:00:00,34040.4,34197.0,34016.9,34188.8,75.479162
3,2023-10-26 21:00:00,34181.7,34212.0,34078.1,34116.2,35.607565
4,2023-10-26 22:00:00,34116.2,34249.0,34116.1,34232.4,49.276334


In [10]:
df.loc[(df['PUBLISHED_ON'] > df_prices.iloc[0, 0]) & (df['PUBLISHED_ON'] < df_prices.iloc[1, 0])]
df.set_index('PUBLISHED_ON', inplace=True)

In [11]:
df.head()

,SENTIMENT
PUBLISHED_ON,
2023-10-26 17:41:25,1
2023-10-26 17:58:21,0
2023-10-26 18:00:00,0
2023-10-26 18:00:00,1
2023-10-26 18:00:12,1


In [12]:
# группируем по индексу, так как есть неуникальные индексы -> склеиваем их
df_unique = df.groupby(level=0)['SENTIMENT'].mean()

In [13]:
df_unique.head(30)

,SENTIMENT
PUBLISHED_ON,
2023-10-26 17:41:25,1.0
2023-10-26 17:58:21,0.0
2023-10-26 18:00:00,0.5
2023-10-26 18:00:12,1.0
2023-10-26 18:03:48,0.0
2023-10-26 18:08:39,0.0
2023-10-26 18:19:03,0.0
2023-10-26 18:20:01,1.0
2023-10-26 18:28:31,1.0


In [14]:
df_prices.iloc[0, 0]

Timestamp('2023-10-26 18:00:00')

In [15]:
def n_moments(n):
    df_full[f'open_prev_{n}_pct'] = df_full['open'].pct_change(periods=n)
    # не сдвигаем, так как используем close-to-open, то есть торгуем от закрытой, про которую все знаем
    df_full[f'high_prev_{n}_pct'] = df_full['high'].pct_change(periods=n)
    df_full[f'low_prev_{n}_pct'] = df_full['low'].pct_change(periods=n)
    df_full[f'close_prev_{n}_pct'] = df_full['close'].pct_change(periods=n)
    df_full[f'volume_prev_{n}_pct'] = df_full['volume'].pct_change(periods=n)

будем работать со средними сентиментами по всем таймфреймам до 1 недели

In [16]:
df_full = df_prices.copy()

N = [1, 4, 12, 24, 168]

sent1 = df_unique.rolling('1h').mean()
sent2 = df_unique.rolling('4h').mean()
sent3 = df_unique.rolling('12h').mean()
sent4 = df_unique.rolling('24h').mean()
sent5 = df_unique.rolling('168h').mean()

for n in N:
    n_moments(n)

In [17]:
sent1.head()

,SENTIMENT
PUBLISHED_ON,
2023-10-26 17:41:25,1.000
2023-10-26 17:58:21,0.500
2023-10-26 18:00:00,0.500
2023-10-26 18:00:12,0.625
2023-10-26 18:03:48,0.500


In [18]:
df_full.dropna(inplace=True)

In [19]:
df_full.set_index('timestamp', inplace=True)

In [20]:
df_full.head()

,open,high,low,close,volume,open_prev_1_pct,high_prev_1_pct,low_prev_1_pct,close_prev_1_pct,volume_prev_1_pct,...,open_prev_24_pct,high_prev_24_pct,low_prev_24_pct,close_prev_24_pct,volume_prev_24_pct,open_prev_168_pct,high_prev_168_pct,low_prev_168_pct,close_prev_168_pct,volume_prev_168_pct
timestamp,,,,,,,,,,,,,,,,,,,,,
2023-11-02 18:00:00,34771.3,34863.0,34723.1,34844.8,138.545733,0.003692,0.002386,0.004940,0.002117,-0.137383,...,0.007461,0.003760,0.008311,0.007765,-0.210943,0.024357,0.022855,0.023133,0.025354,0.580635
2023-11-02 19:00:00,34844.9,35015.8,34809.9,34997.7,127.193183,0.002117,0.004383,0.002500,0.004388,-0.081941,...,0.007765,0.008578,0.009152,0.012721,-0.176514,0.025357,0.026606,0.026565,0.028125,-0.013072
2023-11-02 20:00:00,34999.6,35098.8,34825.1,34902.6,151.208686,0.004440,0.002370,0.000437,-0.002717,0.188811,...,0.012776,-0.009904,0.008321,-0.013644,-0.718599,0.028178,0.026371,0.023759,0.020878,1.003317
2023-11-02 21:00:00,34902.6,34970.0,34850.0,34919.9,50.577013,-0.002771,-0.003670,0.000715,0.000496,-0.665515,...,-0.013647,-0.017109,-0.007091,-0.013041,-0.892516,0.021090,0.022156,0.022651,0.023558,0.420401
2023-11-02 22:00:00,34920.0,34927.6,34766.1,34789.5,51.014521,0.000499,-0.001212,-0.002407,-0.003734,0.008650,...,-0.013041,-0.017035,-0.012102,-0.014023,-0.583462,0.023561,0.019814,0.019053,0.016274,0.035274


1. В чем проблема, которую мы решаем?
sent_1h (Новости): У тебя новости выходят в случайное время. Например, в 10:02, потом в 10:17, потом в 11:45.

df_full (Свечи): Свечи идут строго по сетке. Например, 10:00, 10:05, 10:10, 10:15.

Если ты просто попытаешься вставить колонку новостей в таблицу свечей, Pandas выдаст ошибку, потому что время не совпадает. Свече в 10:05 нужно знать, какое настроение было на рынке, но у тебя нет записи ровно на 10:05. Есть запись на 10:02.

2. Что делает .reindex(df_full.index)?
Эта команда говорит: "Возьми серию данных sent_1h и насильно перестрой её так, чтобы её временны́е метки (индекс) стали точь-в-точь такими же, как у df_full (твоих свечей)."

Но тут возникает вопрос: "Эй, в данных новостей нет строки для времени 10:05. Что мне туда поставить? Пустоту (NaN)?"

3. Что делает method='ffill'?
Это сокращение от Forward Fill (Заполнение вперед). Это ответ на вопрос выше: "Если ты не находишь точного совпадения по времени, не ставь пустоту. Посмотри в прошлое и возьми самое последнее известное значение."

In [21]:
df_full['sentiment_1h'] = sent1.reindex(df_full.index, method='ffill')
df_full['sentiment_4h'] = sent2.reindex(df_full.index, method='ffill')
df_full['sentiment_12h'] = sent3.reindex(df_full.index, method='ffill')
df_full['sentiment_24h'] = sent4.reindex(df_full.index, method='ffill')
df_full['sentiment_168h'] = sent5.reindex(df_full.index, method='ffill')

In [22]:
display(df_full.head())

,open,high,low,close,volume,open_prev_1_pct,high_prev_1_pct,low_prev_1_pct,close_prev_1_pct,volume_prev_1_pct,...,open_prev_168_pct,high_prev_168_pct,low_prev_168_pct,close_prev_168_pct,volume_prev_168_pct,sentiment_1h,sentiment_4h,sentiment_12h,sentiment_24h,sentiment_168h
timestamp,,,,,,,,,,,,,,,,,,,,,
2023-11-02 18:00:00,34771.3,34863.0,34723.1,34844.8,138.545733,0.003692,0.002386,0.004940,0.002117,-0.137383,...,0.024357,0.022855,0.023133,0.025354,0.580635,0.727273,0.659574,0.694215,0.698925,0.588457
2023-11-02 19:00:00,34844.9,35015.8,34809.9,34997.7,127.193183,0.002117,0.004383,0.002500,0.004388,-0.081941,...,0.025357,0.026606,0.026565,0.028125,-0.013072,0.272727,0.571429,0.671875,0.688172,0.586746
2023-11-02 20:00:00,34999.6,35098.8,34825.1,34902.6,151.208686,0.004440,0.002370,0.000437,-0.002717,0.188811,...,0.028178,0.026371,0.023759,0.020878,1.003317,0.666667,0.600000,0.666667,0.690217,0.590811
2023-11-02 21:00:00,34902.6,34970.0,34850.0,34919.9,50.577013,-0.002771,-0.003670,0.000715,0.000496,-0.665515,...,0.021090,0.022156,0.022651,0.023558,0.420401,0.416667,0.526316,0.647059,0.666667,0.591935
2023-11-02 22:00:00,34920.0,34927.6,34766.1,34789.5,51.014521,0.000499,-0.001212,-0.002407,-0.003734,0.008650,...,0.023561,0.019814,0.019053,0.016274,0.035274,0.600000,0.437500,0.625954,0.648352,0.592134


In [23]:
len(df_full)

18742

RSI и ATR. Без них сентимент работает плохо (ты можешь купить на хорошей новости, но на самом пике цены, и тебя прокатит вниз).

In [24]:
!pip install -q pandas-ta

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 9.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.


ATR — это индикатор волатильности. Он не говорит о направлении движения цены (вверх или вниз), а показывает, насколько сильно цена «дергается» в среднем за период.
Измерение: Выражается в тех же единицах, что и цена (например, в долларах или пунктах).
Как читать:
Высокий ATR: На рынке высокая нервозность, свечи большие, цена летает.
Низкий ATR: Затишье, флэт, цена стоит на месте.
Зачем нужен:
Для установки Stop-Loss (если волатильность высокая, стоп-лосс нужно ставить подальше).
Чтобы понять, живой ли сейчас рынок.
Как считается:
Он берет максимум из трех расстояний:
(High - Low) текущей свечи.
(High - Close предыдущей).
(Low - Close предыдущей).
Затем эти значения усредняются (обычно за 14 дней).


In [25]:
import pandas_ta as ta

In [26]:
df_full['ATR'] = df_full.ta.atr(length=14)

RSI (Relative Strength Index) — Индекс относительной силы
RSI — это осциллятор, который измеряет скорость и изменение цены. Он показывает, насколько сильно актив «перекуплен» или «перепродан».
Шкала: от 0 до 100.
Как читать:
Выше 70: Актив «перекуплен» (цена слишком быстро росла, возможен разворот вниз).
Ниже 30: Актив «перепродан» (цена слишком сильно упала, возможен отскок вверх).
Зачем нужен: Помогает найти точки входа и выхода, когда тренд выдыхается.

In [27]:
df_full['RSI'] = df_full.ta.rsi(length=14)

In [28]:
df_full.dropna(inplace=True)

In [29]:
df_full.head()

,open,high,low,close,volume,open_prev_1_pct,high_prev_1_pct,low_prev_1_pct,close_prev_1_pct,volume_prev_1_pct,...,low_prev_168_pct,close_prev_168_pct,volume_prev_168_pct,sentiment_1h,sentiment_4h,sentiment_12h,sentiment_24h,sentiment_168h,ATR,RSI
timestamp,,,,,,,,,,,,,,,,,,,,,
2023-11-03 07:00:00,34448.2,34510.5,34358.1,34468.8,55.249083,-0.002661,-0.001505,-0.001911,0.000595,-0.069608,...,0.012647,0.015407,-0.147280,0.333333,0.357143,0.440000,0.618497,0.589785,218.207143,64.497216
2023-11-03 08:00:00,34468.8,34610.2,34242.8,34427.2,143.105284,0.000598,0.002889,-0.003356,-0.001207,1.590184,...,0.008844,0.010123,2.106160,0.600000,0.529412,0.444444,0.613636,0.588710,228.863776,63.011628
2023-11-03 09:00:00,34427.3,34443.6,34196.6,34257.1,81.204159,-0.001204,-0.004814,-0.001349,-0.004941,-0.432557,...,0.003383,0.002822,1.025482,0.500000,0.500000,0.487805,0.607955,0.586652,230.159220,57.209087
2023-11-03 10:00:00,34257.1,34279.2,34120.0,34279.0,150.981240,-0.004944,-0.004773,-0.002240,0.000639,0.859280,...,0.001600,0.006137,2.179838,0.500000,0.444444,0.459459,0.585366,0.584783,225.090704,57.748552
2023-11-03 11:00:00,34279.0,34280.5,34161.7,34254.0,107.074259,0.000639,0.000038,0.001222,-0.000729,-0.290811,...,0.005584,0.006216,0.437831,0.368421,0.483871,0.470588,0.568047,0.582703,217.498511,56.867190


In [30]:
len(df_full)

18729

In [31]:
# Triple Barrier

- Класс 1: "Цена пошла сильно вверх".Если модель предсказывает 1 $\rightarrow$ Встаем в Лонг.Тейк ставим на Close + 2ATR.Стоп ставим на Close - 2ATR.

- Класс -1: "Цена пошла сильно вниз".Если модель предсказывает -1 $\rightarrow$ Встаем в Шорт.Тейк ставим на Close - 2ATR (потому что в шорте прибыль внизу).Стоп ставим на Close + 2ATR (убыток вверху).

- Класс 0: "Боковик или пила".Не входим.

In [32]:
# Настройки стратегии (Risk/Reward = 2:1)
profit_atr_mult = 2.0
stop_atr_mult = 1.0
horizon = 12

targets = []

for i in tqdm(range(len(df_full) - horizon)):

    current_close = df_full.iloc[i]['close']
    current_atr = df_full.iloc[i]['ATR']

    # Уровни
    long_tp = current_close + (profit_atr_mult * current_atr)
    long_sl = current_close - (stop_atr_mult * current_atr)

    short_tp = current_close - (profit_atr_mult * current_atr)
    short_sl = current_close + (stop_atr_mult * current_atr)

    label = 0

    # Флаги активности (пока стоп не выбит — мы в игре)
    long_active = True
    short_active = True

    for j in range(1, horizon + 1):
        future_idx = i + j
        f_high = df_full.iloc[future_idx]['high']
        f_low = df_full.iloc[future_idx]['low']

        # --- ЛОНГ ---
        if long_active:
            if f_low <= long_sl:
                long_active = False # Стоп выбит, лонг умер
            elif f_high >= long_tp:
                label = 1
                break # Победа лонга

        # --- ШОРТ ---
        if short_active:
            if f_high >= short_sl:
                short_active = False # Стоп выбит, шорт умер
            elif f_low <= short_tp:
                label = -1
                break # Победа шорта

        # Если оба направления выбиты стопами — выходим досрочно, ловить нечего
        if not long_active and not short_active:
            break

    targets.append(label)

# Заполняем хвост
for _ in range(horizon):
    targets.append(np.nan)

df_full['target'] = targets
df_full.dropna(inplace=True)
print(f"Баланс классов:\n{df_full['target'].value_counts()}")

  0%|          | 0/18717 [00:00<?, ?it/s]

Баланс классов:
target
 0.0    9165
 1.0    4837
-1.0    4715
Name: count, dtype: int64


In [33]:
np.nan

nan

In [34]:
len(df_full)

18717

In [35]:
df_full

,open,high,low,close,volume,open_prev_1_pct,high_prev_1_pct,low_prev_1_pct,close_prev_1_pct,volume_prev_1_pct,...,close_prev_168_pct,volume_prev_168_pct,sentiment_1h,sentiment_4h,sentiment_12h,sentiment_24h,sentiment_168h,ATR,RSI,target
timestamp,,,,,,,,,,,,,,,,,,,,,
2023-11-03 07:00:00,34448.2,34510.5,34358.1,34468.8,55.249083,-0.002661,-0.001505,-0.001911,0.000595,-0.069608,...,0.015407,-0.147280,0.333333,0.357143,0.440000,0.618497,0.589785,218.207143,64.497216,0.0
2023-11-03 08:00:00,34468.8,34610.2,34242.8,34427.2,143.105284,0.000598,0.002889,-0.003356,-0.001207,1.590184,...,0.010123,2.106160,0.600000,0.529412,0.444444,0.613636,0.588710,228.863776,63.011628,0.0
2023-11-03 09:00:00,34427.3,34443.6,34196.6,34257.1,81.204159,-0.001204,-0.004814,-0.001349,-0.004941,-0.432557,...,0.002822,1.025482,0.500000,0.500000,0.487805,0.607955,0.586652,230.159220,57.209087,1.0
2023-11-03 10:00:00,34257.1,34279.2,34120.0,34279.0,150.981240,-0.004944,-0.004773,-0.002240,0.000639,0.859280,...,0.006137,2.179838,0.500000,0.444444,0.459459,0.585366,0.584783,225.090704,57.748552,1.0
2023-11-03 11:00:00,34279.0,34280.5,34161.7,34254.0,107.074259,0.000639,0.000038,0.001222,-0.000729,-0.290811,...,0.006216,0.437831,0.368421,0.483871,0.470588,0.568047,0.582703,217.498511,56.867190,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-21 23:00:00,88477.8,88822.9,88378.2,88650.0,189.848137,0.003566,0.003698,0.002916,0.001949,0.058577,...,0.005524,-0.616440,0.000000,-0.029412,-0.050000,0.095491,0.216032,361.637895,57.570414,1.0
2025-12-22 00:00:00,88650.0,89621.8,88615.2,88623.2,428.284779,0.001946,0.008994,0.002682,-0.000302,1.255934,...,0.001682,-0.064708,0.000000,-0.029412,-0.050000,0.095491,0.216032,407.706617,56.907803,0.0
2025-12-22 01:00:00,88622.5,89278.8,88521.4,88630.5,277.885352,-0.000310,-0.003827,-0.001059,0.000082,-0.351167,...,-0.006982,-0.317893,0.000000,-0.029412,-0.050000,0.095491,0.216032,432.684715,57.052803,0.0


12:00:00 — Свеча закрылась. У нас зафиксировались High, Low, Close за период 11:00-12:00.

12:00:01 — Мы мгновенно считаем ATR и RSI по этим зафиксированным данным (это будет ATR последней строки нашего датафрейма).

12:00:02 — Мы подаем этот ATR в модель. Модель говорит "Long".

12:00:03 — Мы посылаем ордер на биржу и входим в рынок по цене Open новой свечи (12:00-13:00).

Вывод: Твоя модель учится предсказывать будущее, стоя в конце текущей свечи. Поэтому в реальном времени ты будешь подавать ей данные только что закрывшейся свечи.

In [36]:
df_full.to_csv('/content/gdrive/MyDrive/Colab Notebooks/course/datasets/df_full_last_version_1.csv')